# Demo: Dataset Middleware

This notebook is the positive condition for [demo-baseline-ontology-extraction.ipynb](./demo-baseline-ontology-extraction.ipynb).
Instead of asking the Research Agent to compose Turtle directly in its final answer, we give it middleware-owned RDF state plus dataset tools and then inspect the resulting graph.

---

- **What changed relative to the baseline?** The task is the same, but graph construction now happens through `DatasetMiddleware` rather than through direct final-answer serialization.
- **Why is that a meaningful change?** It separates graph building from final chat formatting, gives us a machine-readable state we can inspect during and after the run, and makes syntax robustness part of the middleware condition rather than a matter of luck.
- **What additional control is present in this notebook?** The checked-in run uses namespace whitelisting, including an explicit `urn:ex:` extension for local terms, so the agent can only write into a constrained vocabulary surface.
- **How should this notebook be read after the baseline?** The baseline established that prompt-only extraction can fail before semantic evaluation even begins. This notebook asks what improves, and what still does not, once RDF state is handled through middleware.

## 1. Setup

We first filter a LangChain-internal warning that is unrelated to the experiment.



In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers).
What matters for this demo is that Deepagents receives a chat model that can issue tool calls.

_NOTE:_ The checked-in run again follows the Ministral model guidance used in the baseline notebook.
`MinistralPromptSuffixMiddleware` remains enabled so that differences between notebooks are driven by dataset affordances rather than by a silent model-prompt regression.



In [2]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    seed=42,
    temperature=1,
)

## 2. The Demonstration

### 2.1. Prompt Setup

The system prompt stays close to the baseline, but there is one deliberate difference: we no longer inject `DATASET_TIPS` into the shared prompt text.

That removal is methodological.
In the baseline, `DATASET_TIPS` served as tool-agnostic guidance extracted from the middleware prompt so that the negative control was not artificially under-instructed.
Here, dataset-specific guidance belongs to the middleware itself.

What this condition **does** receive:

- The shared RDF modeling guidance from `CORE_PROMPT`.
- The shared vocabulary-selection guidance from `VOCABULARY_TIPS`.
- Middleware-owned dataset tools and middleware-managed RDF session state.
- Namespace whitelisting that constrains where new terms may be written.

What this condition **does not** receive:

- A requirement to place the final answer itself in valid Turtle.
- Arbitrary namespace freedom beyond the whitelist and the explicit local `urn:ex:` extension.

This is the core comparison point with the baseline.
The question is no longer whether the model can typeset Turtle in one shot.
The question is whether tool-mediated graph construction gives us a more inspectable and more reliable extraction process.



In [3]:
from typing import Final

from demo_utils import CORE_PROMPT, STOPPING_CRITERIA, VOCABULARY_TIPS

# Reduce prompt asymmetry without recreating middleware-only capabilities.
SYSTEM_PROMPT: Final[str] = CORE_PROMPT + VOCABULARY_TIPS + STOPPING_CRITERIA

### 2.2. Agent Task Definition

We keep the task itself fixed across the series so that the intervention remains the middleware composition rather than a rewritten assignment.
The Research Agent still receives the same short passage and the same request to formalize it as RDF.

What changes is the execution path.
Instead of treating the final chat message as the only artifact that matters, we can now observe the graph that the agent constructed through tools.
That distinction matters because the stored dataset is the authoritative output of this condition.



In [4]:
from demo_utils import TASK

print(TASK)


Please assist me in representing the subject matter of the following text as an RDF graph.
Please use <urn:ex:> as the base for any IRIs that you need to mint as part
of your response. Be sure to label any new terms or properties that you mint
so that they are human readable.

```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, too. Emotions and theological views aside, can we formalize this?
```



#### 2.2.1 Evaluation Contract

The target graph is intentionally unchanged from the baseline notebook.
That lets us compare the two conditions without shifting the success criterion.

The middleware condition also changes the order of evaluation.
Here, parseability is no longer gated on the model emitting one perfect Turtle block in its final answer.
Instead, we can evaluate the graph captured in middleware-managed state and treat the final chat response as secondary.

Best-practice documentation triples remain acceptable extras.
The aim is to compare extracted graph content, not to punish the run for providing human-readable labels or comments.



In [5]:
from demo_utils import expected_graph
from rdflib import Graph

print(expected_graph.serialize(format="turtle"))

@prefix ex: <urn:ex:> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Haplorhini a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Primate .

ex:Hominidae a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Haplorhini .

ex:Hominina a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominidae .

ex:HomoSapiens a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominina .

ex:Mammal a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Animal .

ex:Primate a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Mammal .

ex:John a ex:Person .

ex:Person rdfs:subClassOf ex:HomoSapiens ;
    owl:equivalentClass ex:HomoSapiens .




### 2.3. Tracing the Agent

One benefit of this setup is observability.
`LiveNotebookTrace` lets us inspect the Research Agent's actions while `DatasetMiddleware` accumulates RDF state behind the scenes.
That combination makes the run auditable in a way that the baseline condition is not: we can see not just the outcome, but the structured side effects that produced it.

The middleware configuration in this notebook also includes namespace whitelisting.
That is important for interpretation because it turns vocabulary control into an explicit system property instead of a prompt-level suggestion.
The agent can still mint local terms under `urn:ex:`, but it cannot wander freely across arbitrary namespaces.

In the output, we see some noteworthy examples of the controls afforded to the agent:

1. Turn 1 - `add_triples` rejects a plain-text object and explains how to resubmit it as an RDF literal.
2. Turn 2 - The agent corrects the literal encoding and `add_triples` accepts all three triples.
3. Turn 4 - The vocabulary whitelist rejects `rdfs:Property` and suggests the allowed `rdf:Property` alternative.
4. Turn 5 - The agent switches to `rdf:Property`, and the corrected call succeeds without repeating the whitelist violation.

In [6]:
from deepagents import create_deep_agent
from langgraph.errors import GraphRecursionError
from rdflib import Namespace
from rdflib_reasoning.middleware.dataset_middleware import (
    DatasetMiddleware,
    DatasetMiddlewareConfig,
)
from rdflib_reasoning.middleware.ministral_middleware import (
    MinistralPromptSuffixMiddleware,
)
from rdflib_reasoning.middleware.namespaces.spec_whitelist import (
    RestrictedNamespaceWhitelist,
)
from rdflib_reasoning.middleware.tracing_notebook import LiveNotebookTrace

dataset_middleware = DatasetMiddleware(
    config=DatasetMiddlewareConfig(
        namespace_whitelist=RestrictedNamespaceWhitelist().plus_entries(
            ("ex", Namespace("urn:ex:"))
        )
    )
)

with LiveNotebookTrace(heading="Dataset Trace") as trace:
    agent = trace.attach(
        create_deep_agent(
            model=llm,
            system_prompt=SYSTEM_PROMPT,
            middleware=[
                dataset_middleware,
                MinistralPromptSuffixMiddleware(),
            ],
        )
    )

    RUN_INPUT = {"messages": [{"role": "user", "content": TASK}]}
    # RUN_CONFIG = {"recursion_limit": 60}
    RUN_CONFIG = {}
    try:
        agent_response = agent.invoke(RUN_INPUT, config=RUN_CONFIG)
    except GraphRecursionError:
        display(trace.sink.snapshot())
        raise

# Dataset Trace

## Turn 1

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Person"
    },
    {
      "object": "Person",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Person"
    },
    {
      "object": "A biological classification for humans.",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Person"
    }
  ]
}
```

#### Rejection

```json
{
  "content": "Error invoking tool 'add_triples' with kwargs {'triples': [{'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'object': 'http://www.w3.org/2000/01/rdf-schema#Class'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#label', 'object': 'Person'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#comment', 'object': 'A biological classification for humans.'}]} with error:\n triples.2.object: Value error, Could not parse RDF term from A biological classification for humans.. If this is plain text, encode it as an RDF literal like \"\\\"A biological classification for humans.\\\"\". If this is an IRI, provide it either in canonical N3 form like <urn:example:Thing> or as a bare RFC 3987 IRI.\n Please fix the error and try again.",
  "status": "error",
  "tool_call_id": "M35IihxPo"
}
```

## Turn 2

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"Person\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Person"
    },
    {
      "object": "\"A biological classification for humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Person"
    }
  ]
}
```

#### Result

```json
updated=3 message='3 of 3 triples added.'
```

## Turn 3

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"Homo Sapiens\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "\"The biological species classification for modern humans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"Hominina\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "\"A subtribe within the Hominini tribe, which includes modern humans and their direct ancestors.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominina"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"Hominidae\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "\"The family of great apes, which includes modern humans, chimpanzees, gorillas, and orangutans.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"Haplorhini\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Haplorhini"
    },
    {
      "object": "\"The suborder of primates that includes humans, apes, and tarsiers.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Haplorhini"
    }
  ]
}
```

#### Result

```json
updated=12 message='12 of 12 triples added.'
```

## Turn 4

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"Primate\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"The order of mammals that includes humans, apes, monkeys, and prosimians.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"Mammal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"A class of vertebrate animals that includes humans and other animals that nurse their young with milk and have hair or fur.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"Animal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"A kingdom of living organisms that includes humans and other multi-cellular organisms characterized by movement and reproduction.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Property",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"subtribe of\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"Relates a taxonomic group to its subtribe.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:subtribeOf"
    }
  ]
}
```

#### Rejection

```json
{
  "content": "The term http://www.w3.org/2000/01/rdf-schema#Property is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples` again with this same IRI.\n\nSuggested alternatives (Levenshtein distance measures string similarity; lower is better):\n\n| Qualified name | IRI | Levenshtein distance |\n| --- | --- | ---: |\n| `rdf:Property` | `http://www.w3.org/1999/02/22-rdf-syntax-ns#Property` | 1 |\n| `owl:onProperty` | `http://www.w3.org/2002/07/owl#onProperty` | 6 |\n\nIf none of the suggestions fit your intent, you MUST use a different term from the allowed vocabularies listed in the system prompt.",
  "status": "error",
  "tool_call_id": "VuuWG8DYt"
}
```

## Turn 5

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/1999/02/22-rdf-syntax-ns#Property",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"subtribe of\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "\"Relates a taxonomic group to its subtribe.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:subtribeOf"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"Primate\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "\"The order of mammals that includes humans, apes, monkeys, and prosimians.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Primate"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"Mammal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "\"A class of vertebrate animals that includes humans and other animals that nurse their young with milk and have hair or fur.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Mammal"
    },
    {
      "object": "http://www.w3.org/2000/01/rdf-schema#Class",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"Animal\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:Animal"
    },
    {
      "object": "\"A kingdom of living organisms that includes humans and other multi-cellular organisms characterized by movement and reproduction.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:Animal"
    }
  ]
}
```

#### Result

```json
updated=3 message='3 of 12 triples added. 9 of 12 triples already existed.'
```

## Turn 6

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "http://www.w3.org/1999/02/22-rdf-syntax-ns#Property",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:classifiedAs"
    },
    {
      "object": "\"classified as\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:classifiedAs"
    },
    {
      "object": "\"Relates a group or individual to its biological classification.\"",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#comment",
      "subject": "urn:ex:classifiedAs"
    }
  ]
}
```

#### Result

```json
updated=3 message='3 of 3 triples added.'
```

## Turn 7

### Tool: add_triples

#### Arguments

```json
{
  "triples": [
    {
      "object": "urn:ex:Person",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:John"
    },
    {
      "object": "John",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#label",
      "subject": "urn:ex:John"
    },
    {
      "object": "urn:ex:Person",
      "predicate": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
      "subject": "urn:ex:ModernPeople"
    },
    {
      "object": "urn:ex:HomoSapiens",
      "predicate": "urn:ex:classifiedAs",
      "subject": "urn:ex:ModernPeople"
    },
    {
      "object": "urn:ex:Hominina",
      "predicate": "urn:ex:subtribeOf",
      "subject": "urn:ex:HomoSapiens"
    },
    {
      "object": "urn:ex:Haplorhini",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "subject": "urn:ex:Hominidae"
    },
    {
      "object": "urn:ex:Hominina",
      "predicate": "urn:ex:contains",
      "subject": "urn:ex:Hominidae"
    }
  ]
}
```

#### Result

```json
updated=7 message='7 of 7 triples added.'
```

## Turn 8

### Final Response

> I will define the property `ex:contains` with a label and comment and also represent the relationship between primates, mammals, and animals as specified in the text. To do this, I will:
> 1. Add a definition for `ex:contains`.
> 2. Represent `ex:Primate` as a subclass of `ex:Mammal`.
> 3. Represent `ex:Mammal` as a subclass of `ex:Animal`.
>
> Let's proceed with these additions.

### Tool: add_triples

#### Rejection

```json
{
  "content": "The term http://www.w3.org/2000/01/rdf-schema#Property is not in the vocabulary whitelist for this knowledge base. You SHOULD verify that you are using the correct term or consider the alternatives below. You MUST NOT call `add_triples` again with this same IRI.\n\nSuggested alternatives (Levenshtein distance measures string similarity; lower is better):\n\n| Qualified name | IRI | Levenshtein distance |\n| --- | --- | ---: |\n| `rdf:Property` | `http://www.w3.org/1999/02/22-rdf-syntax-ns#Property` | 1 |\n| `owl:onProperty` | `http://www.w3.org/2002/07/owl#onProperty` | 6 |\n\nIf none of the suggestions fit your intent, you MUST use a different term from the allowed vocabularies listed in the system prompt.",
  "status": "error",
  "tool_call_id": "VuuWG8DYt"
}
```

### Tool: add_triples

#### Rejection

```json
{
  "content": "Error invoking tool 'add_triples' with kwargs {'triples': [{'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'object': 'http://www.w3.org/2000/01/rdf-schema#Class'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#label', 'object': 'Person'}, {'subject': 'urn:ex:Person', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#comment', 'object': 'A biological classification for humans.'}]} with error:\n triples.2.object: Value error, Could not parse RDF term from A biological classification for humans.. If this is plain text, encode it as an RDF literal like \"\\\"A biological classification for humans.\\\"\". If this is an IRI, provide it either in canonical N3 form like <urn:example:Thing> or as a bare RFC 3987 IRI.\n Please fix the error and try again.",
  "status": "error",
  "tool_call_id": "M35IihxPo"
}
```


### 2.4. Inspecting the Resulting Dataset

The next cell snapshots the middleware-owned dataset directly.
This is the key practical difference from the baseline notebook: we do not need to recover the graph from the model's final prose formatting, because the graph already exists as structured state.

The checked-in run illustrates both the strength and the limit of this condition.
It successfully leaves behind a readable dataset with classes, labels, comments, and a small number of relation terms.
That is already a better operational outcome than the baseline run, which failed before semantic comparison was even possible.

At the same time, inspectability makes the remaining problems easier to name.
The graph includes invented relation terms such as `classifiedAs`, `contains`, and `subtribeOf`, and some taxonomic links are represented in idiosyncratic ways.
So middleware improves robustness and debuggability here, but it does not by itself solve vocabulary grounding or ontology design.

### 2.5. Accessing the Knowledge Base

We retained a reference to the middleware when creating the agent:

```python
dataset_middleware = DatasetMiddleware(...)
```

That means we can snapshot the dataset at any point, including after the run, without asking the Research Agent to serialize RDF correctly on demand.
For experiments, this is a useful separation of concerns: tool-mediated state capture and human-readable reporting no longer have to be the same step.



In [7]:
from rdflib import Dataset

dataset: Dataset = dataset_middleware.session.snapshot_dataset()

print("There are", len(dataset), "triples in the dataset")
print("-" * 80)
print(dataset.serialize(format="turtle"))

There are 37 triples in the dataset
--------------------------------------------------------------------------------
@prefix ns1: <urn:ex:> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ns1:Animal a rdfs:Class ;
    rdfs:label "Animal" ;
    rdfs:comment "A kingdom of living organisms that includes humans and other multi-cellular organisms characterized by movement and reproduction." .

ns1:Haplorhini a rdfs:Class ;
    rdfs:label "Haplorhini" ;
    rdfs:comment "The suborder of primates that includes humans, apes, and tarsiers." .

ns1:Hominidae a rdfs:Class ;
    rdfs:label "Hominidae" ;
    rdfs:comment "The family of great apes, which includes modern humans, chimpanzees, gorillas, and orangutans." ;
    rdfs:subClassOf ns1:Haplorhini ;
    ns1:contains ns1:Hominina .

ns1:Hominina a rdfs:Class ;
    rdfs:label "Hominina" ;
    rdfs:comment "A subtribe within the Hominini tribe, which includes modern humans an

### 2.6. Metrics

Because the dataset is already parseable RDF, semantic comparison can proceed directly in this notebook.
That is the first major contrast with the baseline condition, where evaluation stopped at the parseability gate.

The current run still requires careful interpretation.
Our target graph is small and opinionated, while the middleware snapshot contains both target-like assertions and additional documentation structure.
So the metrics tell us something important, but not everything.

#### 2.6.1 Interpreting Missing Facts

Missing target triples still matter.
In this run they mostly indicate that the Research Agent chose local modeling patterns instead of reusing the target relations and class structure exactly.
That is easier to diagnose now because the dataset is available for inspection, but the underlying problem remains one of grounding and modeling choice rather than of syntax.

Several of the misses are therefore informative rather than mysterious:

1. Some expected subclass links are absent because the agent introduced alternative local predicates.
2. Some entities are described, labeled, or typed without being connected using the target relation.
3. The whitelist prevents arbitrary namespace sprawl, but it does not tell the agent which allowed term is the right one to use.

#### 2.6.2 Interpreting Extra Facts

Extra triples are especially important to read qualitatively in this condition.
Many of them are documentation or schema-support triples that make the dataset easier to inspect, such as labels, comments, and local property declarations.
Those extras are not evidence of failure by themselves.

Other extras reveal modeling drift.
When the agent mints a relation like `classifiedAs` or `subtribeOf`, it may still be trying to express something intelligible, but it is no longer aligning with the target graph.
That distinction is exactly why dataset inspection and graph-level traces are useful experimental artifacts.

#### 2.6.3 What the Current Metrics Show

The checked-in run clears the operational hurdle that blocked the baseline: we have a machine-usable graph and can compute overlap metrics.
The resulting scores are still far from an exact match, with both substantial misses and non-trivial extras.

That combination supports a narrower claim than "the middleware solved extraction."
A better reading is:

1. `DatasetMiddleware` makes the extraction process operationally reliable enough to inspect and score.
2. Namespace whitelisting keeps the writable vocabulary surface controlled.
3. Meaningful semantic alignment still needs stronger grounding than dataset mutation alone provides.



In [8]:
from demo_utils import evaluate_actual_graph

actual_graph: Graph = dataset.default_graph
measures, graphs = evaluate_actual_graph(actual_graph)

intersection = graphs["intersection"]
missing = graphs["missing"]
extra = graphs["extra"]
union = graphs["union"]

In [9]:
import sys

if measures["counts"]["intersection"] > 0:
    print(intersection.serialize(format="turtle"))
else:
    print(
        "The agent's graph output did NOT intersect the expected graph.",
        file=sys.stderr,
    )

@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:ex:Haplorhini> a rdfs:Class .

<urn:ex:Hominidae> a rdfs:Class ;
    rdfs:subClassOf <urn:ex:Haplorhini> .

<urn:ex:Hominina> a rdfs:Class .

<urn:ex:HomoSapiens> a rdfs:Class .

<urn:ex:Mammal> a rdfs:Class .

<urn:ex:Primate> a rdfs:Class .

<urn:ex:John> a <urn:ex:Person> .




In [10]:
print(missing.serialize(format="turtle"))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<urn:ex:Haplorhini> a owl:Class ;
    rdfs:subClassOf <urn:ex:Primate> .

<urn:ex:Person> rdfs:subClassOf <urn:ex:HomoSapiens> ;
    owl:equivalentClass <urn:ex:HomoSapiens> .

<urn:ex:Hominidae> a owl:Class .

<urn:ex:Hominina> a owl:Class ;
    rdfs:subClassOf <urn:ex:Hominidae> .

<urn:ex:Mammal> a owl:Class ;
    rdfs:subClassOf <urn:ex:Animal> .

<urn:ex:Primate> a owl:Class ;
    rdfs:subClassOf <urn:ex:Mammal> .

<urn:ex:HomoSapiens> a owl:Class ;
    rdfs:subClassOf <urn:ex:Hominina> .




In [11]:
print(extra.serialize(format="turtle"))

@prefix ns1: <urn:ex:> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ns1:Animal a rdfs:Class .

ns1:Person a rdfs:Class .

ns1:Hominidae ns1:contains ns1:Hominina .

ns1:ModernPeople a ns1:Person ;
    ns1:classifiedAs ns1:HomoSapiens .

ns1:classifiedAs a rdf:Property .

ns1:subtribeOf a rdf:Property .

ns1:HomoSapiens ns1:subtribeOf ns1:Hominina .




In [12]:
from demo_utils import pprint_measures

pprint_measures(measures)

Expected triples: 21
Actual triples: 16
Intersection triples: 8
Missing triples: 13
Extra triples: 8
Union triples: 29
Precision: 0.500
Recall: 0.381
F1 score: 0.432
Jaccard Index: 0.276
Triple Edit Distance: 21
Normalized Triple Edit Distance: 0.724
Exact Match: False


## 3. Conclusions

This notebook demonstrates a real improvement over the baseline condition, but not a complete one.
With `DatasetMiddleware`, the Research Agent leaves behind structured RDF state that we can inspect and evaluate even when final-answer formatting would otherwise be brittle.
That makes the run more observable, more controllable, and more useful for comparison.

The checked-in whitelisted run also shows the boundary of that improvement.
Constraining namespace usage and moving graph construction into middleware does not automatically yield the intended ontology.
The agent still invents local relations and misses parts of the target structure.

That outcome is informative.
It suggests that dataset-backed state is a strong foundation for reliable experimentation, while vocabulary discovery and stronger grounding remain separate capabilities to add on top.

